# 01 · Exploración y transformación — de Bronze a Silver

**Preguntas que se contestan en esta capa:** ¿cuántas líneas de pedido son válidas? ¿qué problemas de calidad tenían los datos y cuántas filas afectaban cada uno?

## Qué pasa en este notebook

Silver es la capa donde los datos dejan de ser "tal como llegaron" y pasan a ser confiables. El trabajo sigue el mismo orden que se vio en el curso 

1. **Perfil de calidad.** Se cuentan nulos, duplicados y valores fuera de rango en Bronze. Ese perfil define qué reglas de limpieza necesita cada tabla.
2. **Limpieza y tipado.**
   - `productos`: las tallas se normalizan a `XS, S, M, L, XL`; los colores nulos se marcan como `Sin dato`; el precio, que venía como texto con `$`, se convierte a número.
   - `clientes`: las ciudades nulas pasan a `Desconocida`; se descartan el nombre y el correo, porque el análisis no los necesita y así los datos personales se quedan solo en Bronze.
   - `lineas_pedido`: se eliminan duplicados por `linea_id`, se descartan las filas sin cliente o producto y las de cantidad cero, y el precio, que venía como texto con coma decimal, se convierte a número.
3. **Etiqueta todavía en cero.** Cada línea entra a Silver con `fue_devuelto = 0` y `fecha_devolucion` nula, porque las devoluciones aún no han llegado. Se aplican en el siguiente notebook con un `MERGE INTO`.

In [0]:
%run ./00_Setup/00_Configuracion

**En esta celda** se importa la caja de herramientas de PySpark: `F` para las expresiones (`when`, `coalesce`, casts), los tipos para `fecha_devolucion`, `Window` para las ventanas y `broadcast` para los joins. Es la última celda de preparación: a partir de aquí el notebook es análisis puro.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast

**En esta celda** se mide el problema antes de limpiarlo: el perfil de calidad de `lineas_pedido`, la tabla central del proyecto. Tres técnicas: (1) nulos por columna en una sola pasada con `F.sum(col.isNull().cast(int))` — un solo job en vez de una consulta por columna; (2) duplicados comparando el total de filas contra los `linea_id` distintos; (3) violaciones de rango con `filter()`. Cada número que imprime esta celda se convierte en una regla de limpieza en la celda 15.

In [0]:
df_lineas_bronze = spark.table(tbl(SCHEMA_BRONZE, "lineas_pedido"))
total_lineas = df_lineas_bronze.count()

print("PERFIL DE CALIDAD: lineas_pedido (Bronze)")
print("=" * 60)
print(f"Total de registros: {total_lineas:,}")

print("\n1. Nulos por columna:")
display(df_lineas_bronze.select([
    F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
    for c in ["cliente_id", "producto_id", "cantidad", "precio_pagado"]
]))

print("\n2. Duplicados por clave natural (linea_id):")
unicos = df_lineas_bronze.select("linea_id").distinct().count()
print(f"   Registros: {total_lineas:,}  |  linea_id unicos: {unicos:,}  |  Duplicados: {total_lineas - unicos:,}")

print("\n3. Valores fuera de rango y formatos:")
n_cantidad = df_lineas_bronze.filter("cantidad <= 0").count()
n_coma = df_lineas_bronze.filter("precio_pagado LIKE '%,%'").count()
print(f"   cantidad <= 0                  : {n_cantidad:,}")
print(f"   precio con coma decimal (texto): {n_coma:,}")

**En esta celda** se perfila la calidad de las dos dimensiones, `productos` y `clientes`. El hallazgo más importante es la tabla de tallas: si `MEDIUM` y `Medium` cuentan como valores distintos, el modelo verá categorías que en realidad son la misma. También se cuantifican los colores y ciudades nulos y los precios con `$`, que delatan columnas de texto donde debería haber números.

In [0]:
df_productos_bronze = spark.table(tbl(SCHEMA_BRONZE, "productos"))
df_clientes_bronze  = spark.table(tbl(SCHEMA_BRONZE, "clientes"))

print("PERFIL DE CALIDAD: productos y clientes (Bronze)")
print("=" * 60)
print("Variantes de talla encontradas:")
display(df_productos_bronze.groupBy("talla").count().orderBy("talla"))

n_color_nulo  = df_productos_bronze.filter("color IS NULL").count()
n_precio_signo = df_productos_bronze.filter("precio_lista LIKE '$%'").count()
n_ciudad_nula = df_clientes_bronze.filter("ciudad IS NULL").count()
print(f"productos con color nulo      : {n_color_nulo:,}")
print(f"productos con precio con '$'  : {n_precio_signo:,}")
print(f"clientes con ciudad nula      : {n_ciudad_nula:,}")

**En esta celda** se escribe la primera tabla de Silver: `productos`. La talla se normaliza a mayúsculas sin espacios y los nombres largos se traducen (`MEDIUM` → `M`) con una cascada de `F.when()`, para que el modelo vea 5 tallas y no el doble de categorías por el mismo concepto. El precio se limpia con `regexp_replace` (solo quedan dígitos y el punto) y se convierte con `try_cast`, que devuelve nulo en vez de fallar: un nulo se detecta en el perfil, un valor corrupto no. La tabla sale deduplicada por `producto_id` y con `color` nulo reemplazado por `Sin dato`.

In [0]:
talla_limpia = F.upper(F.trim(F.col("talla")))

df_productos_silver = (
    df_productos_bronze
    .dropDuplicates(["producto_id"])
    .withColumn("talla",
        F.when(talla_limpia == "EXTRA SMALL", "XS")
         .when(talla_limpia == "SMALL", "S")
         .when(talla_limpia == "MEDIUM", "M")
         .when(talla_limpia == "LARGE", "L")
         .when(talla_limpia == "EXTRA LARGE", "XL")
         .otherwise(talla_limpia))
    .withColumn("color", F.coalesce(F.col("color"), F.lit("Sin dato")))
    .withColumn("precio_lista", F.expr("try_cast(regexp_replace(precio_lista, '[^0-9.]', '') AS DOUBLE)"))
    .select("producto_id", "nombre", "categoria", "talla", "color", "precio_lista")
)

escribir_delta(df_productos_silver, SCHEMA_SILVER, "productos", "Productos limpios: talla normalizada, precio numerico")
display(df_productos_silver.groupBy("talla").count().orderBy("talla"))

**En esta celda** se escribe `silver.clientes`, con una decisión de gobernanza además de la limpieza: el nombre y el correo se dejan fuera porque el análisis no los necesita. Minimizar los datos personales desde el diseño reduce la superficie de riesgo, y esos campos quedan solo en Bronze, la capa que no se consulta directamente. La única limpieza de datos es la ciudad nula, que pasa a `Desconocida` para no perder filas en los joins de Gold.

In [0]:
df_clientes_silver = (
    df_clientes_bronze
    .dropDuplicates(["cliente_id"])
    .withColumn("ciudad", F.coalesce(F.col("ciudad"), F.lit("Desconocida")))
    .select("cliente_id", "ciudad", "segmento", "fecha_registro")
)

escribir_delta(df_clientes_silver, SCHEMA_SILVER, "clientes", "Clientes sin datos personales (sin nombre ni correo)")

**En esta celda** se limpia la tabla central del proyecto y se escribe `silver.lineas_pedido`. Cada regla responde a un hallazgo del perfil de la celda 7: duplicados por `linea_id`, filas sin cliente o producto, cantidades en cero o negativas, y el precio en texto con coma decimal que pasa a número. Cada línea entra con `fue_devuelto = 0` y `fecha_devolucion` nula: la devolución ocurre *después* de la compra, así que la etiqueta se completa en 03_Delta_Lake con un `MERGE INTO` — este es el corazón del problema de ML del proyecto: predecir un evento que todavía no ha ocurrido.

**Por qué `mes` como partición:** particionar por `fecha` generaría 365 carpetas de unas 55 filas — demasiados archivos chicos, justo lo que Delta compacta después (se verá con `OPTIMIZE` en 03_Delta_Lake). Con `mes` son 12 particiones con miles de filas cada una, y los filtros por mes siguen aprovechando la partición.

In [0]:
df_lineas_silver = (
    df_lineas_bronze
    .dropDuplicates(["linea_id"])
    .filter(F.col("cliente_id").isNotNull() & F.col("producto_id").isNotNull() & (F.col("cantidad") > 0))
    .withColumn("precio_pagado", F.expr("try_cast(regexp_replace(precio_pagado, ',', '.') AS DOUBLE)"))
    .filter(F.col("precio_pagado") > 0)
    .withColumn("fue_devuelto", F.lit(0))                                  # todavia no llegan las devoluciones
    .withColumn("fecha_devolucion", F.lit(None).cast(DateType()))
    .select("linea_id", "pedido_id", "cliente_id", "producto_id", "fecha", "mes", "cantidad",
            "descuento_pct", "precio_pagado", "fue_devuelto", "fecha_devolucion")
)

escribir_delta(df_lineas_silver, SCHEMA_SILVER, "lineas_pedido",
               "Lineas de pedido limpias. fue_devuelto se actualiza con MERGE", partition_col="mes")

**En esta celda** se cierra el notebook con su conclusión: el balance de la limpieza. Se compara Bronze contra Silver para responder cuántas filas se descartaron y por qué, y es también la evidencia de que la arquitectura Medallion aporta algo: si una regla resultara demasiado agresiva, el dato original sigue intacto en Bronze y Silver se puede rehacer desde cero. El conteo con `fue_devuelto = 0` confirma que esta etapa termina exactamente donde debe: con la etiqueta lista para el `MERGE INTO` de 03_Delta_Lake.

In [0]:
bronze_n = spark.table(tbl(SCHEMA_BRONZE, "lineas_pedido")).count()
silver_n = spark.table(tbl(SCHEMA_SILVER, "lineas_pedido")).count()

print("=" * 60)
print("BRONZE vs SILVER — lineas_pedido")
print("=" * 60)
print(f"  Bronze (tal como llego)      : {bronze_n:,}")
print(f"  Silver (limpias y validas)   : {silver_n:,}")
print(f"  Filas descartadas            : {bronze_n - silver_n:,} ({(bronze_n - silver_n) / bronze_n:.1%})")
print("  Motivos: duplicados por linea_id, cliente o producto nulo, cantidad <= 0.")
print("\nLas devoluciones todavia no se aplicaron: fue_devuelto = 0 en todas las lineas.")
display(spark.table(tbl(SCHEMA_SILVER, "lineas_pedido")).groupBy("fue_devuelto").count())

✅ Silver completa: `productos`, `clientes` y `lineas_pedido` limpias y validadas. &nbsp;·&nbsp; Avance del proyecto: configuración → ingesta Bronze → **limpieza Silver (aquí)** → Delta Lake → Gold → ML.